In [1]:
import pandas as pd

In [2]:
df = pd.read_excel("Kompensatsiya-ishga-joylashgan_joylashmagan (2).xlsx")


In [ ]:
df

In [ ]:
# 1. Store duplicates in separate df for reference
duplicates_df = df[df.duplicated(subset=['F.I.Sh'], keep=False)]

# 2. Drop duplicates keeping first occurrence
df_clean = df.drop_duplicates(subset=['F.I.Sh'], keep='first')

# 3. Verify results
print(f"Original rows: {len(df)}")
print(f"Rows after removing duplicates: {len(df_clean)}")
print(f"Number of duplicates removed: {len(df) - len(df_clean)}")

# Optional: Save clean DataFrame
df_clean.to_csv('clean_data.csv', index=False)

In [7]:
df_new = pd.read_csv('ishga-joylashganlar.csv')

In [ ]:
df_new

In [ ]:
from telethon.sync import TelegramClient
from telethon.tl.functions.messages import GetRepliesRequest
import pandas as pd
import asyncio
import nest_asyncio
from datetime import datetime

# Enable nested event loops
nest_asyncio.apply()

# Create unique session name
session_name = f'session_{datetime.now().timestamp()}'

api_id = '20037141'
api_hash = '0e98ba0ebb245fa54b7afce53b7117c1'

async def fetch_and_save_comments():
    async with TelegramClient(session_name, api_id, api_hash) as client:
        await client.start()
        
        channel = 'uzbekcoders_uzb'
        post_id = 11

        # Get the post and its comments
        post = await client.get_messages(channel, ids=post_id)
        
        # Initialize list to store comments
        comments_data = []
        
        # Fetch replies/comments
        async for reply in client.iter_messages(channel, reply_to=post_id):
            try:
                # Convert timezone-aware datetime to timezone-naive
                naive_date = reply.date.replace(tzinfo=None)
                
                comment_data = {
                    'comment_id': reply.id,
                    'user_id': reply.from_id.user_id if reply.from_id else None,
                    'username': (await client.get_entity(reply.from_id.user_id)).username if reply.from_id else None,
                    'text': reply.text,
                    'date': naive_date,  # Use timezone-naive datetime
                    'reply_to': reply.reply_to_msg_id if reply.reply_to else None
                }
                comments_data.append(comment_data)
                print(f"Fetched comment: {reply.text[:50]}...")
            except Exception as e:
                print(f"Error processing comment: {str(e)}")
                continue

        # Create DataFrame
        df = pd.DataFrame(comments_data)
        
        # Save to Excel
        excel_filename = f'telegram_comments_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'
        df.to_excel(excel_filename, index=False)
        print(f"\nComments saved to {excel_filename}")
        print(f"Total comments fetched: {len(comments_data)}")

# For Jupyter/IPython environment
loop = asyncio.get_event_loop()
loop.run_until_complete(fetch_and_save_comments())